# Weekly player research

Reproducible replacement for the historical distribution portion of `Who-Should-I-Start.ipynb`. Uses **synthetic demonstration data**, no network or database. This notebook describes observed games; use the weekly planner for start/sit forecasts, injury context and roster constraints. Install the package with `pip install -e ".[viz]"` and select that environment as the notebook kernel.

In [ ]:
from pathlib import Path
import pandas as pd
from fantasyfootball.reports.visualization import PlotlyVisualizer

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "examples").is_dir()), None)
if root is None:
    raise RuntimeError("Run this notebook inside the fantasy-football repository")
source = root / "examples" / "weekly_chart.csv"
weekly = pd.read_csv(source, dtype={"player_id": str})
season = 2026
player_ids = ["demo-rb", "demo-wr"]
selected = weekly.loc[weekly.season.eq(season) & weekly.player_id.isin(player_ids)].copy()
if selected.empty:
    raise ValueError("No observations match the selected season and players")
charts = PlotlyVisualizer()
selected

## Weekly production

Missing weeks are gaps, not zero scores. Data must contain one observed game per player/week, with points already scored using the same league rules. Do not insert inactive or bye weeks as zero. Names are labels; IDs identify players.

In [ ]:
trend = charts.weekly_trends(selected)
trend

## Empirical distribution

The vertical axis is the fraction of supplied games with points at or below the horizontal value. Each jump is observed data, with ties combined. This is not a smoothed density or a prediction interval. Sample counts appear in the tooltip and table below. Unequal samples, changing roles and missing games affect interpretation; no DNP probability is inferred.

In [ ]:
distribution = charts.weekly_distribution(selected)
distribution

In [ ]:
summary = selected.groupby(["player_id", "player_name"], sort=True).fantasy_points.agg(games="count", mean="mean", median="median", minimum="min", maximum="max")
summary

## Optional offline export

Set `export = True` to write a local HTML report under ignored `work/`. It embeds Plotly once and includes the observed data table. No source CSV is modified.

In [ ]:
export = False
if export:
    charts.write_weekly_report(selected, root / "work" / "weekly-research.html")
